# Pairwise alignments

---
## Learning Objectives

1. Conceptually undersand dynamic programming and sequence alignment
1. Implement Smith-Waterman algorithm for local alignment


---
## Background

This week we will be implementing Smith-Waterman. This is a dynamic programming algorithm used for local sequence alignment. 

As a reminder, the scoring for Smith-Waterman only uses the scores from the positions above, left, and above-left of the current position in the matrix as below:

<center><img src="figures/Smith-Waterman_scoring.png"></center>

For traceback, you will need to keep track of the direction of the arrows in a matrix and then begin traceback from the maximum value.

# Smith-Waterman Algorithm Implementation Activity
## Introduction

Pairwise sequence alignment is a fundamental technique in bioinformatics used to compare two biological sequences, such as DNA, RNA, or proteins. It helps identify regions of similarity that may indicate functional, structural, or evolutionary relationships between the sequences.

The brute force method for sequence alignment, which involves comparing every possible alignment, is highly inefficient. For two sequences of length m and n, the time complexity would be \(T(m,n) = O(mn \cdot 2^{m+n})\), making it impractical for longer sequences.

Dynamic programming offers a more efficient solution. It breaks down the problem into smaller subproblems and stores their solutions to avoid redundant computations. This approach reduces the time complexity to \(O(mn)\).

The Smith-Waterman algorithm is a dynamic programming approach for local sequence alignment. It identifies the optimal local alignment between two sequences by comparing all possible pairs of segments from the sequences and finding the best-scoring alignment.

## Activity Overview

In this activity, you will implement the Smith-Waterman algorithm for local sequence alignment. The algorithm consists of two main parts:

Scoring matrix calculation
Traceback for optimal alignment reconstruction

The pseudocode for the Smith-Waterman algorithm is as follows:

$$
\begin{aligned}
& \text{Initialize scoring matrix } H \text{ with zeros} \\
& \text{For } i \text{ from } 1 \text{ to } m: \\
&     \text{For } j \text{ from } 1 \text{ to } n: \\
&         H[i][j] = \max \begin{cases}
&             0 \\
&             H[i-1][j-1] + \text{match/mismatch score} \\
&             H[i-1][j] + \text{gap penalty} \\
&             H[i][j-1] + \text{gap penalty}
&         \end{cases} \\
& \text{Find the highest score in } H \text{ and its position} \\
& \text{Perform traceback from the highest score position} \\
& \text{Return the optimal local alignment}
\end{aligned}
$$

## Instructions

Working in pairs, implement the Smith-Waterman algorithm using Python. Your implementation should include:

1. A function to calculate the scoring matrix (`cal_score`)
2. A function to perform the traceback and reconstruct the optimal alignment (`traceback`)
3. A dynamic scoring strategy that allows for customizable gap penalties and match/mismatch scores (`smith_waterman`)

## Follow these steps:

### Implement the scoring matrix calculation function:

1. Initialize the matrix with zeros
2. Fill the matrix using the Smith-Waterman algorithm
3. Return the completed matrix and the position of the highest score

### Implement the traceback function

1. Start from the highest score position
2. Trace back through the matrix to reconstruct the optimal alignment
3. Return the aligned sequences and the alignment score

### Implement a main function that:

1. Takes two sequences as input
2. Accepts parameters for match score, mismatch penalty, and gap penalty
3. Calls the scoring and traceback functions
4. Prints the aligned sequences and the alignment score

Test your implementation with various input sequences and scoring parameters

---
## Imports

In [1]:
import numpy as np

---
## Implement Smith-Waterman algorithm


```
SmithWaterman(seq1, seq2, match, mismatch, gap)
    Initialize [len(seq1)+1] x [len(seq2)+1] numpy array as scoring matrix with first column and row equal to 0
    
    Fill scoring matrix (score_matrix) and Traceback matrix, record the position with max score (max_pos):
    for i in each row number:
        for j in each column number:
            S[i][j] = max( S[i-1][j-1] + compute_diag_score, S[i-1][j] + gap_score, H[i][j-1] + gap_score, 0 )
            T[i][j] = direction of max( S[i-1][j-1] + compute_diag_score, S[i-1][j] + gap_score, S[i][j-1] + gap_score, 0 )
    
    Traceback. Find the optimal path through scoring matrix starting at max_pos
    
```

In [ ]:
def cal_score(matrix, seq1, seq2, i, j, match, mismatch, gap):
    '''Calculate score for position (i,j) in scoring matrix, also record move to trace back
    
    Args:
        matrix (numpy array): scoring matrix
        seq1 (str): sequence 1
        seq2 (str): sequence 2
        i (int): current row number
        j (int): current column number
        
    Returns:
        score in position (i,j)    
        move to trace back: 0-END, 1-DIAG, 2-UP, 3-LEFT
        
    Pseudocode:
        Calculate scores based on upper-left, up, and left neighbors:
            diag_score = upper-left + (match or mismatch)
            up_score = up + gap
            left_score = left + gap
        score = max(0, diag_score, up_score, left_score)
        traceback = maximum direction or end

    pseudocode:
        get the 2 letters for this cell
        check if they are a match or a mismatch
            use this to calculate diagonal score
        calculate up and left scores by adding gap penalty to previous cell
        take the max of diagnoal, left and up scores with 0 as the floor,
            diagonal gets checked first, then up, then left (diagonal first to ensure least amount of gaps)
            if up and left are tied, move will always be up; this is an arbitrary but consistent bias, switching the elif statements for these 2 would flip the bias 
        decide the traceback move
            if max is above 0, record the score and the direction it came from
            if max is 0 or negative, record 0 and end the traceback

        
    '''
    if seq1[i-1] == seq2[j-1]: #need -1 offset because row i in matrix is letter i-1 in sequence
        diag = matrix[i-1][j-1] + match
    else:
        diag = matrix[i-1][j-1] + mismatch
    above = matrix[i-1][j] + gap
    left = matrix[i][j-1] + gap
    maximum = max(0, diag, above, left)
    if maximum > 0:
        if maximum == diag:
            move = 1 # 1 = diagonal
        elif maximum == above:
            move = 2 # 2 = up
        elif maximum == left:
            move = 3 # 3 = left
    else:
        move = 0 # 0 = end
    return maximum, move

In [ ]:
def traceback(seq1, seq2, traceback_matrix, maximum_position):
    '''Find the optimal path through scoring marix
        
        diagonal: match/mismatch
        up: gap in seq2
        left: gap in seq1
        
    Args:
        seq1 (str) : First sequence being aligned
        seq2 (str) : Second sequence being aligned
        traceback_matrix (numpy array): traceback matrix
        maximum_position (tuple): starting position to trace back from
        
    Returns:
        aligned_seq1 (str): e.g. GTTGAC
        aligned_seq2 (str): e.g. GTT-AC
        
    Pseudocode:
        while current_move != END:
            current_move = traceback_matrix[current_row][current_col]
            if current_move == DIAG:
                ...
            elif current_move == UP:
                ...
            elif current_move == LEFT:
                ...
            
    pseudocode:
    get the starting row and column from the maximum_position argument
    initialize 2 empty strings aligned_seq1 and aligned_seq2 for the aligned sequences
    initialize starting move with a non-zero value so the while loop can check for 0
    while current_move is not 0
        read the move stored at the current cell
            if 1(diagonal): add both letters, move up and left 1 row and column
            if 2(up): add a letter to seq1 and seq2 gets a "-", move up 1 row
            if 3(left): add a letter to seq2 and seq1 gets a "-", move left 1 column
    return the 2 aligned sequences
    '''
    current_row, current_column = maximum_position
    aligned_seq1 = ""
    aligned_seq2 = ""
    current_move = -1 #set to non-zero value so it can pass the while loop, set to -1 because the real move codes are 0, 1, 2, 3 so it's a clear placeholder
    while current_move != 0:
        current_move = traceback_matrix[current_row, current_column]
        if current_move == 1: #if there is a match / mismatch
            aligned_seq1 = seq1[current_row - 1] + aligned_seq1 #record sequence 1 nucleotide
            aligned_seq2 = seq2[current_column - 1] + aligned_seq2 #record sequence 2 nucleotide
            current_row = current_row - 1 #set current row to the previous one
            current_column = current_column - 1 #set current column to the previous one, ultiamtely setting the current row and column to the previous diagonal 
        elif current_move == 2: #if there is a gap (up)
            aligned_seq1 = seq1[current_row - 1] + aligned_seq1 #record sequence 1 nucleotide
            aligned_seq2 = "-" + aligned_seq2 #record sequence 2 gap
            current_row = current_row - 1
        elif current_move == 3: #if there is a gap (left)
            aligned_seq1 = "-" + aligned_seq1 # record sequence 1 gap
            aligned_seq2 = seq2[current_column - 1] + aligned_seq2 #record sequence 2 nucleotide
            current_column = current_column - 1
    return aligned_seq1, aligned_seq2
            

In [ ]:
def smith_waterman(seq1, seq2, match=1, mismatch=-1, gap=-1):
    '''Smith-Waterman algorithm for local alignment
    
    Args:
        seq1 (str): input seq 1
        seq2 (str): input seq 2
        match: default = +1
        mismatch: default = -1
        gap: default = -1
    
    Returns:
        aligned_seq1 (str)
        aligned_seq2 (str)
        score_matrix (numpy array): scoring matrix
    
    psuedocode
    initialize score matrix of zeros, size has to be (len(seq1) + 1) x (len(seq2) + 1)
    create traceback matrix of zeros, also the same size
    for each row i from 1 to len(seq1)
        for each column j from 1 to len(seq2)
            call cal_score for each cell (i,j)
            store the score in the score matrix at (i,j)
            store the move in the traceback matrix at (i,j)    
    find the position (row, column) of the highest score in score matrix
    call traceback with seq1, seq2, traceback matrix and that position
    return the 2 aligned sequences and the score matrix
    '''
    score_matrix = np.zeros((len(seq1) + 1, len(seq2) + 1), dtype=int) #initialize score_matrix with zeros
    traceback_matrix = np.zeros((len(seq1)+ 1, len(seq2) + 1), dtype = int) #initialize traceback_matrix with zeros
    for i in range(1, len(seq1)+1):
        for j in range(1, len(seq2) + 1):
            score, move = cal_score(score_matrix, seq1, seq2, i, j, match, mismatch, gap)
            score_matrix[i,j] = score #put score into score_matrix at this position
            traceback_matrix[i,j] = move #put move into traceback_matrix at this position

    maximum_position = np.unravel_index(np.argmax(score_matrix), score_matrix.shape) #use .shape to determine size of array, which is used by np.unravel_index to convert np.argmax single value into a row and column
    aligned_seq1, aligned_seq2 = traceback(seq1, seq2, traceback_matrix, maximum_position) #call traceback function and assign outputs to variables
    return aligned_seq1, aligned_seq2, score_matrix

In [5]:
# Example from slides
seq1 = 'TACTTAG'
seq2 = 'CACATTAA'

aligned_seq1, aligned_seq2, score_matrix = smith_waterman(seq1,seq2)

print (aligned_seq1)
print (aligned_seq2)
print (score_matrix)

AC-TTA
ACATTA
[[0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 1. 1. 0. 0.]
 [0. 0. 1. 0. 1. 0. 0. 2. 1.]
 [0. 1. 0. 2. 1. 0. 0. 1. 1.]
 [0. 0. 0. 1. 1. 2. 1. 0. 0.]
 [0. 0. 0. 0. 0. 2. 3. 2. 1.]
 [0. 0. 1. 0. 1. 1. 2. 4. 3.]
 [0. 0. 0. 0. 0. 0. 1. 3. 3.]]


In [6]:
seq3 = "GGGGAAAAAAAAAAAACCCC"
seq4 = "GGGGAAAAAACCCC"

aligned_seq3, aligned_seq4, score_matrix = smith_waterman(seq3,seq4)

print (aligned_seq3)
print (aligned_seq4)
print (score_matrix)

GGGGAAAAAA
GGGGAAAAAA
[[ 0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.]
 [ 0.  1.  1.  1.  1.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.]
 [ 0.  1.  2.  2.  2.  1.  0.  0.  0.  0.  0.  0.  0.  0.  0.]
 [ 0.  1.  2.  3.  3.  2.  1.  0.  0.  0.  0.  0.  0.  0.  0.]
 [ 0.  1.  2.  3.  4.  3.  2.  1.  0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  1.  2.  3.  5.  4.  3.  2.  1.  1.  0.  0.  0.  0.]
 [ 0.  0.  0.  1.  2.  4.  6.  5.  4.  3.  2.  1.  0.  0.  0.]
 [ 0.  0.  0.  0.  1.  3.  5.  7.  6.  5.  4.  3.  2.  1.  0.]
 [ 0.  0.  0.  0.  0.  2.  4.  6.  8.  7.  6.  5.  4.  3.  2.]
 [ 0.  0.  0.  0.  0.  1.  3.  5.  7.  9.  8.  7.  6.  5.  4.]
 [ 0.  0.  0.  0.  0.  1.  2.  4.  6.  8. 10.  9.  8.  7.  6.]
 [ 0.  0.  0.  0.  0.  1.  2.  3.  5.  7.  9.  9.  8.  7.  6.]
 [ 0.  0.  0.  0.  0.  1.  2.  3.  4.  6.  8.  8.  8.  7.  6.]
 [ 0.  0.  0.  0.  0.  1.  2.  3.  4.  5.  7.  7.  7.  7.  6.]
 [ 0.  0.  0.  0.  0.  1.  2.  3.  4.  5.  6.  6.  6.  6.  6.]
 [ 0.  0.  0.  0.  0.  1.  2.  3.